In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd
train=pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test=pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

print(train.shape)
print(test.shape)
train.head()

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = train[features]
y = train["Survived"]

final_test= test[features]
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 数值列和类别列
num_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_features = ["Sex", "Embarked"]

# 数值预处理：填缺失值 + 标准化
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())#标准化
])

# 类别预处理：填缺失值 + OneHot
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),#众数
    ("onehot", OneHotEncoder(handle_unknown="ignore"))#文字变成0 1，then ignore unkonw
])

# 合并预处理
preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features)
])

# 模型
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("模型", RandomForestClassifier(n_estimators=200, random_state=42))
])

# 训练
model.fit(X, y)

print("训练完成")
#====================
#X_processed = preprocessor.fit_transform(X)
#print(type(X_processed))
#print(X_processed.shape)
#feature_names = preprocessor.get_feature_names_out()
#X_processed_df = pd.DataFrame(X_processed, columns=feature_names)
#print(X_processed_df)
#preproess 的表
#====================
final_predict=model.predict(final_test)
print("train successed")
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': final_predict.astype(int)
})
submission.to_csv('submission.csv', index=False, encoding='utf-8')
print("submission.csv")

(891, 12)
(418, 11)
训练完成
train successed
submission.csv
